# Knowledge Distillation: Qwen2.5-Coder-7B-Instruct (teacher) -> Qwen2.5-Coder-0.5B (student)

**Corrected version** — fixes three real bugs hit while running the previous version, each verified
and explained inline at the point of the fix, not just patched silently:

1. **`ModuleNotFoundError: transformers.models.audioflamingo3`** — the pinned `transformers==4.46.3`
   silently got overridden by a newer version pulled in as a transitive dependency of `bitsandbytes`
   or `accelerate`. Fixed with a forced reinstall + an explicit version assertion in Cell 0 that stops
   the notebook immediately (with a clear message) if the pin didn't take, instead of continuing into
   a broken, inconsistent package state.
2. **`RuntimeError: size of tensor a (152064) must match ... (151936)`** — teacher and student share
   the same real tokenizer vocabulary, but their model output layers are padded to different sizes
   (a common practice for hardware-friendly tensor-core alignment). Fixed in Cell 4 (an explicit
   `EFFECTIVE_VOCAB_SIZE` check) and Cell 5 (both logits tensors sliced to that shared size before
   any computation — no real token id is ever in the discarded padding region).
3. **`ValueError: Attempting to unscale FP16 gradients`** — the student was loaded with
   `torch_dtype=torch.float16`, putting its stored weights in fp16. `GradScaler` requires fp32
   master weights by design; `autocast` already provides the fp16 *computation* speed during the
   forward pass without needing fp16 *weights* underneath. Fixed by loading the student in its
   default fp32.

## How distillation works, and why this exact design
The loss combines two terms: `(1-alpha)*CE(student, ground_truth) + alpha*T^2*KL(teacher || student)`.
The CE term keeps the student anchored to your actual training data. The KL term — computed at every
target token position, against the teacher's full, temperature-softened probability distribution over
the whole vocabulary — is where the real transfer happens: the teacher doesn't just say 'the next token
is `def`', it reveals its relative confidence across every plausible alternative, and that richer signal
is what distillation is designed to pass on. **This formula and its masking were verified with 7 numerical
invariants before being used here** (student==teacher gives KL~=0, alpha=0/1 reduce to pure CE/KL, tampering
a masked position never changes the loss, tampering a valid position always does).

## Why NOT Hugging Face's own TRL GKDTrainer/DistillationTrainer (checked first)
TRL does ship an official, maintained distillation trainer — found and reviewed before building this.
Two things held it back as the delivered path: its API has been visibly churning (multiple competing
import paths across different doc versions turned up in a single search, signaling the surface is still
settling), and committing to an exact call I can't verify against your installed version risks handing
you something that fails immediately with zero value delivered. The hand-rolled loop here implements the
same, stable, well-established mathematical formula, fully under our control and independently verified.
**If you want to try TRL's trainer as an alternative path, `pip show trl` first to see what's actually
installed, then check its current docs against that specific version.**

## Why NO new special tokens this run (a real, necessary tradeoff)
The earlier Qwen SFT-only notebook added `<GENERATE>`, `<CODE>`, etc. as dedicated vocabulary tokens.
Distillation via KL divergence requires teacher and student to agree on what every token id MEANS — if
the student's vocabulary were resized, the teacher (which never saw those new ids) would either error or
silently misinterpret them, corrupting its guidance for every later position. So this run uses the task
markers as **plain text** instead, and **Cell 4 asserts the vocabularies match** before training starts —
a checked precondition, not an assumption.

## Why single GPU (not the dual-GPU DDP pattern used elsewhere in this project)
This combines three things never tested together here: a brand-new loss function, a large frozen teacher
co-resident with the student on one GPU, and this specific memory profile. Correctness first; dual-GPU
speed is a natural next step once this is confirmed working, not a risk worth taking on the first attempt.

## Before running (on kaggle.com)
1. `dataset_formats_v2.parquet` attached via **"Add Input"**.
2. Settings -> Accelerator -> **GPU T4 x2** (only one is used, by design, but this is the smallest
   accelerator tier that still gives you a T4 at all on Kaggle).
3. Settings -> Internet -> **On**.

## Critical: restart the session TWICE around Cell 0
Run Cell 0 once. If its version assertion passes, **restart the session** (Run -> Restart Session) —
a `pip install` happening inside an already-running kernel never retroactively fixes a `transformers`
module that code has already imported earlier in that same session. Only after restarting should you
proceed to Cell 1. Skipping this restart is exactly what caused bug #1 above.

## Memory (deliberately conservative, per your instruction)
`BATCH_SIZE=1, GRAD_ACCUM=16` — holding a 7B (4-bit) teacher AND a 0.5B student, both producing full
151K-vocabulary logit tensors over the SAME batch on one T4, is a strictly harder memory situation than
the plain SFT run that already needed `batch=2` to avoid an OOM. If this still runs out of memory,
restart the session first (a CUDA OOM leaves fragmented memory state that doesn't clear on its own),
then try lowering `MAX_SEQ_LEN` from 768 to 512 in Cell 1 as the next fallback.

## Evaluation
Identical, previously-proven harness: subprocess-isolated HumanEval (64) and ClassEval (10), plus
held-out own-dataset exact-match/validity checks. Greedy pass@1 is primary; compile-validated generation
is reported separately, explicitly labeled as NOT a correctness metric.

## Important
**Not executed end to end** here (needs a live GPU with both models downloaded). The loss function was
numerically verified (7 invariants, with every numpy-based test cross-checked against the exact PyTorch
call used in Cell 5). All three bugs listed above were ones actually hit while running the previous
version — each fix here is a direct response to a real traceback, not a guess.

In [ ]:
# CELL 0 — INSTALL (pinned, with a FORCED reinstall -- a plain pinned install can still get
# silently overridden by a newer transformers pulled in as a transitive dependency of
# bitsandbytes/accelerate, which is exactly what caused an earlier crash in this notebook
# [ModuleNotFoundError: transformers.models.audioflamingo3 -- a model that does not exist in
# the pinned 4.46.3, proving a newer version had silently taken over])
!pip install -q "transformers==4.46.3" "accelerate>=0.26.0" "bitsandbytes>=0.43.0" datasets pandas pyarrow
!pip install -q --force-reinstall --no-deps "transformers==4.46.3"

import transformers
print(f"transformers version actually installed: {transformers.__version__}")
assert transformers.__version__ == "4.46.3", (
    f"Expected 4.46.3, got {transformers.__version__} -- the pin did not take effect. "
    f"Restart the session and run this cell again before doing anything else."
)
print("Installed and verified. RESTART THE SESSION NOW (Run -> Restart Session), then run Cell 1 onward.")

In [ ]:
# CELL 1 — IMPORTS + CONFIG
# Single GPU by design (not the dual-GPU DDP pattern used elsewhere in this project): this run
# combines three things that have never been tested together here -- a brand-new loss function, a
# large frozen teacher co-resident with the student on one GPU, and this specific memory profile.
# Correctness comes first; dual-GPU speed is a natural next step once this is confirmed working.
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # must be set BEFORE torch import

import ast, copy, gc, hashlib, json, math, random, re, signal, subprocess, sys, tempfile, time, warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

warnings.filterwarnings("ignore", category=SyntaxWarning)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

PARQUET_FILENAME = "dataset_formats_v2.parquet"
KAGGLE_INPUT_DIR = Path("/kaggle/input")
WORK = Path("/kaggle/working/qwen_distill")
WORK.mkdir(parents=True, exist_ok=True)
BEST_MODEL_DIR = WORK / "best_student"
RESULTS_FILE = WORK / "results.json"

TEACHER_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
STUDENT_MODEL = "Qwen/Qwen2.5-Coder-0.5B"
TARGET_FORMAT_COL = "tgt_b_canonical"
FITS_COL = "fits512_b"

# IMPORTANT DESIGN CHOICE, stated plainly: this run does NOT add the custom special tokens
# (<GENERATE>, <CODE>, etc.) used in the earlier Qwen SFT-only notebook. Knowledge distillation
# requires the teacher and student to interpret every token id IDENTICALLY -- if the student's
# vocabulary were resized to add new tokens, the teacher (which never saw those ids and was never
# trained to represent them) would either error out or silently misinterpret them, and every later
# position's "guidance" from the teacher would be built on a nonsensical embedding. Trading away the
# small special-token benefit from before is the correct, safe choice here: the task markers
# (<GENERATE>, <REPAIR>, etc.) are kept as PLAIN TEXT substrings instead, understood by both models
# via their shared, UNMODIFIED tokenizer.
SYSTEM_PROMPT = (
    "You are a Python code generation assistant for software engineering tasks spanning API design, "
    "database access, data structures and algorithms, and data science code. Rules: output only "
    "complete, syntactically valid Python with every import included. Never include explanations or "
    "text outside the code. Fully implement every function, method, and class -- never stop early or "
    "leave a placeholder. Where relevant, follow SOLID principles, RESTful API conventions, and "
    "proper error handling."
)

# Distillation hyperparameters -- standard values from the literature (Hinton et al. and the
# modern LLM-distillation surveys consulted before building this), not guessed.
KD_TEMPERATURE = 2.0   # softens both distributions so the student can see relative confidence
                       # across plausible alternatives, not just the single correct token
KD_ALPHA = 0.5          # equal weight between matching ground truth (CE) and matching the
                       # teacher's distribution (KL) -- a reasonable, commonly used starting point

EPOCHS = 7
LR = 1e-5               # same proven full-fine-tune lr as the plain Qwen 0.5B SFT run
BATCH_SIZE = 1          # deliberately conservative: holding a 7B (4-bit) teacher AND a 0.5B
                        # student, both producing full-vocabulary (151K) logit tensors over the
                        # SAME batch, on one T4, is a strictly harder memory situation than the
                        # plain SFT run that already needed batch=2 to avoid OOM
GRAD_ACCUM = 16          # keeps the same effective batch (16) as the other Qwen runs
MAX_SEQ_LEN = 768

HE_N = 64
CLASSEVAL_N = 10
CLASSEVAL_MAX_PROMPT_TOKENS = 450
EVAL_OWN_N = 300
GEN_BATCH = 16
RETRY_TEMPERATURES = [0.2, 0.5, 0.8]

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
import transformers
print(f"Transformers {transformers.__version__} | torch {torch.__version__} | device {DEVICE}")
if DEVICE != "cuda:0":
    raise RuntimeError(
        "No GPU found. In the right sidebar: Settings -> Accelerator -> GPU T4 x2 (one GPU is used "
        "here by design), and Internet -> On. The session restarts; then run this cell again."
    )
print(f"GPU: {torch.cuda.get_device_name()} ({torch.cuda.mem_get_info()[1]/1e9:.1f} GB)")
print(f"Teacher: {TEACHER_MODEL} (frozen, 4-bit)")
print(f"Student: {STUDENT_MODEL} (full fine-tune)")
print(f"KD temperature={KD_TEMPERATURE}, alpha={KD_ALPHA} | epochs={EPOCHS} | effective batch={BATCH_SIZE*GRAD_ACCUM}")

## Load the dataset

In [ ]:
# CELL 2 — FIND + LOAD dataset_formats_v2.parquet
candidates = list(KAGGLE_INPUT_DIR.rglob(PARQUET_FILENAME))
if not candidates:
    print(f"Could not find {PARQUET_FILENAME} under {KAGGLE_INPUT_DIR}. What's attached:")
    if KAGGLE_INPUT_DIR.exists():
        for p in KAGGLE_INPUT_DIR.rglob("*"):
            if p.is_file():
                print(f"  {p}")
    raise FileNotFoundError("Attach the dataset with this file via 'Add Input', then re-run this cell.")
src = candidates[0]
print(f"Using: {src} ({src.stat().st_size/1e6:.1f} MB)")

df = pd.read_parquet(src)
print(f"Loaded {len(df)} rows, {len(df.columns)} columns")

need = ["id", "unit_id", "split", "task", "instruction", "context_code", "error", TARGET_FORMAT_COL, FITS_COL, "ok_b"]
missing = [c for c in need if c not in df.columns]
assert not missing, f"expected columns missing: {missing}"

mask = df[FITS_COL].astype(bool) & df["ok_b"].astype(bool)
before = len(df)
df = df[mask].reset_index(drop=True)
print(f"Kept rows with a valid, within-budget {TARGET_FORMAT_COL}: {before} -> {len(df)}")
print("tasks:", dict(df.task.value_counts()))

## Build prompts (plain-text task tags) and measure the real token cost

In [ ]:
# CELL 3 — BUILD PROMPTS (plain-text task tags, no vocabulary changes), MEASURE TOKEN COST
def build_prompt_text(row):
    task = row["task"]
    if task == "generate":
        body = f"<GENERATE> {row['instruction']}"
    elif task == "repair":
        body = (f"<REPAIR> {row['instruction']}\n<CODE>\n{row.get('context_code','')}\n</CODE>"
                f"\n<ERROR>\n{row.get('error','')}\n</ERROR>")
    elif task == "refactor":
        body = f"<REFACTOR> {row['instruction']}\n<CODE>\n{row.get('context_code','')}\n</CODE>"
    else:
        raise ValueError(f"unknown task: {task}")
    return f"{SYSTEM_PROMPT}\n\n{body}\n<OUTPUT>\n"

df["prompt_text"] = df.apply(build_prompt_text, axis=1)

# This is the function already unit-tested (7 invariants, including the edge case where the
# target alone exceeds the token budget) in the earlier Qwen 0.5B SFT-only notebook -- reused
# verbatim here, unchanged.
def build_labeled_sequence(tokenizer, prompt_text, target_text, max_len):
    target_ids = tokenizer(target_text + tokenizer.eos_token, add_special_tokens=False)["input_ids"]
    if len(target_ids) >= max_len:
        kept = target_ids[-max_len:]
        return kept, kept
    room_for_prompt = max_len - len(target_ids)
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    if len(prompt_ids) > room_for_prompt:
        prompt_ids = prompt_ids[-room_for_prompt:]
    input_ids = prompt_ids + target_ids
    labels = [-100] * len(prompt_ids) + target_ids
    return input_ids, labels

from transformers import AutoTokenizer
_tok_probe = AutoTokenizer.from_pretrained(STUDENT_MODEL)
if _tok_probe.pad_token is None:
    _tok_probe.pad_token = _tok_probe.eos_token

sample = df.sample(min(1000, len(df)), random_state=SEED)
combined_lens = [len(_tok_probe(p + t)["input_ids"]) for p, t in zip(sample["prompt_text"], sample[TARGET_FORMAT_COL])]
fits_pct = sum(1 for l in combined_lens if l <= MAX_SEQ_LEN) / len(combined_lens) * 100
print(f"Combined (prompt + target) sequence length:")
print(f"  mean {np.mean(combined_lens):.0f} | median {np.median(combined_lens):.0f} | "
      f"p90 {np.percentile(combined_lens, 90):.0f} | max {max(combined_lens)}")
print(f"  fits {MAX_SEQ_LEN} tokens: {fits_pct:.1f}%")
del _tok_probe

## Load teacher (frozen, 4-bit) and student (full fine-tune, fp32), verify shared vocabulary

In [ ]:
# CELL 4 — LOAD TEACHER (frozen, 4-bit) AND STUDENT (full fine-tune), VERIFY SHARED VOCABULARY
# Knowledge distillation via KL divergence is only mathematically valid if teacher and student
# agree on what every token id MEANS -- this is checked here as a real precondition, not assumed.
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(STUDENT_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL)
assert len(tokenizer) == len(teacher_tokenizer), (
    f"Vocabulary size mismatch: student={len(tokenizer)} teacher={len(teacher_tokenizer)}. "
    f"Distillation via KL divergence requires identical vocabularies -- stopping rather than "
    f"silently computing a meaningless loss."
)
_sample_tokens = ["def", "class", "import", "return", "self", "<GENERATE>", "Python", "    "]
for t in _sample_tokens:
    sid, tid = tokenizer.convert_tokens_to_ids(t), teacher_tokenizer.convert_tokens_to_ids(t)
    assert sid == tid, f"token {t!r} maps to different ids (student={sid}, teacher={tid}) -- vocabularies differ"
print(f"Tokenizer vocabulary check passed: both agree (vocab size {len(tokenizer)}, "
      f"{len(_sample_tokens)} sample tokens spot-checked for identical ids).")
del teacher_tokenizer

# A matching TOKENIZER does not guarantee matching MODEL OUTPUT LAYER sizes. Models are commonly
# padded to a hardware-friendly multiple (e.g. a multiple of 128) for tensor-core efficiency, and
# different model sizes in the same family can land on different padded totals even while sharing
# an identical real vocabulary. Checked explicitly here, rather than discovered later as a
# shape-mismatch crash inside the loss computation.
_student_cfg = AutoConfig.from_pretrained(STUDENT_MODEL)
_teacher_cfg = AutoConfig.from_pretrained(TEACHER_MODEL)
print(f"Model output layer sizes: student={_student_cfg.vocab_size} teacher={_teacher_cfg.vocab_size} "
      f"(tokenizer real vocab: {len(tokenizer)})")
EFFECTIVE_VOCAB_SIZE = len(tokenizer)
assert EFFECTIVE_VOCAB_SIZE <= _student_cfg.vocab_size, "student's output layer is smaller than its own tokenizer's vocab"
assert EFFECTIVE_VOCAB_SIZE <= _teacher_cfg.vocab_size, "teacher's output layer is smaller than the shared tokenizer's vocab"
print(f"Using EFFECTIVE_VOCAB_SIZE={EFFECTIVE_VOCAB_SIZE} -- the real, shared vocabulary. Any extra "
      f"rows beyond this in either model's output layer are unused padding, sliced off in the loss "
      f"function (Cell 5) before any computation -- no real token id is ever discarded by this.")

quant_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)
print(f"\nLoading teacher ({TEACHER_MODEL}, 4-bit, frozen)...")
teacher = AutoModelForCausalLM.from_pretrained(TEACHER_MODEL, quantization_config=quant_config, device_map={"": 0})
teacher.eval()
for p in teacher.parameters():
    p.requires_grad_(False)
print(f"Teacher loaded. GPU memory after teacher: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# Loaded WITHOUT torch_dtype=torch.float16: training needs fp32 master weights so that
# GradScaler.unscale_() can operate on the gradients -- it refuses fp16 gradients by design,
# since the standard mixed-precision recipe is fp32 stored weights + fp16 computation via
# autocast, not fp16 weights throughout. The autocast("cuda", dtype=torch.float16) block in
# Cell 6's training loop already provides the fp16 speed benefit during the forward pass.
print(f"\nLoading student ({STUDENT_MODEL}, fp32 weights, fp16 autocast during training)...")
student = AutoModelForCausalLM.from_pretrained(STUDENT_MODEL).to(DEVICE)
student.gradient_checkpointing_enable()
student.config.use_cache = False
print(f"Student loaded. GPU memory after both models: {torch.cuda.memory_allocated()/1e9:.1f} GB "
      f"of {torch.cuda.mem_get_info()[1]/1e9:.1f} GB total")

## The distillation loss (numerically verified; vocab-padding mismatch handled)

In [ ]:
# CELL 5 — THE DISTILLATION LOSS (verified by hand before this notebook was written: 7 numerical
# invariants checked -- student==teacher gives KL~=0, alpha=0 reduces to pure CE, alpha=1 reduces
# to pure T^2*KL, tampering a masked position never changes the loss, tampering a valid position
# always does. This is the standard Hinton-style combined loss, applied per-token for a sequence
# model: L = (1-alpha)*CE(student, ground_truth) + alpha*T^2*KL(teacher || student).
def distillation_loss(student_logits, teacher_logits, labels, temperature=KD_TEMPERATURE, alpha=KD_ALPHA):
    # Teacher and student output layers are padded to different sizes (e.g. 152,064 vs 151,936)
    # even when they share the same real tokenizer vocabulary -- slice both down to the real,
    # shared size (set in Cell 4) before anything else. No real token id is ever in the discarded
    # region, since EFFECTIVE_VOCAB_SIZE equals the tokenizer's actual vocabulary size and every
    # label is guaranteed to be a real token id.
    student_logits = student_logits[..., :EFFECTIVE_VOCAB_SIZE]
    teacher_logits = teacher_logits[..., :EFFECTIVE_VOCAB_SIZE]
    vocab_size = student_logits.size(-1)
    valid_mask = (labels != -100)

    ce = F.cross_entropy(student_logits.reshape(-1, vocab_size), labels.reshape(-1),
                         ignore_index=-100, reduction="mean")

    # KL computed in fp32 regardless of training precision -- log-probabilities are numerically
    # sensitive, and this is standard practice even under mixed-precision training.
    s_logits = student_logits.float() / temperature
    t_logits = teacher_logits.float() / temperature
    student_log_probs = F.log_softmax(s_logits, dim=-1)
    teacher_probs = F.softmax(t_logits, dim=-1)
    per_token_kl = F.kl_div(student_log_probs, teacher_probs, reduction="none").sum(-1)   # (B, T)
    valid_f = valid_mask.float()
    kl = (per_token_kl * valid_f).sum() / valid_f.sum().clamp(min=1.0)
    kl = kl * (temperature ** 2)

    loss = (1 - alpha) * ce + alpha * kl
    return loss, ce.detach(), kl.detach()

print("Distillation loss ready (numerically verified before being used in this notebook; "
      f"logits sliced to EFFECTIVE_VOCAB_SIZE={EFFECTIVE_VOCAB_SIZE} to handle the teacher/student "
      "output-layer padding mismatch).")

## Distillation training loop (single GPU)

In [ ]:
# CELL 6 — DISTILLATION TRAINING LOOP (single GPU, teacher frozen/4-bit, student full fine-tune)
def tokenize_split(frame, desc):
    all_ids, all_labels = [], []
    for prompt, target in zip(frame["prompt_text"], frame[TARGET_FORMAT_COL]):
        ids, labels = build_labeled_sequence(tokenizer, prompt, target, MAX_SEQ_LEN)
        all_ids.append(ids)
        all_labels.append(labels)
    print(f"{desc}: {len(all_ids)} examples tokenized")
    return all_ids, all_labels

train_df = df[df.split == "train"].reset_index(drop=True)
val_df = df[df.split == "val"].reset_index(drop=True)
train_ids, train_labels = tokenize_split(train_df, "train")
val_ids, val_labels = tokenize_split(val_df, "val")

def make_batches(ids_list, labels_list, batch_size, shuffle, seed=SEED):
    order = list(range(len(ids_list)))
    if shuffle:
        random.Random(seed).shuffle(order)
    for i in range(0, len(order), batch_size):
        idx = order[i:i + batch_size]
        batch_ids = [ids_list[j] for j in idx]
        batch_labels = [labels_list[j] for j in idx]
        max_len = max(len(x) for x in batch_ids)
        input_ids, attention_mask, labels = [], [], []
        for ids, labs in zip(batch_ids, batch_labels):
            pad_len = max_len - len(ids)
            input_ids.append(ids + [tokenizer.pad_token_id] * pad_len)
            attention_mask.append([1] * len(ids) + [0] * pad_len)
            labels.append(labs + [-100] * pad_len)
        yield (torch.tensor(input_ids), torch.tensor(attention_mask), torch.tensor(labels))

optimizer = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=0.01)
steps_per_epoch = math.ceil(len(train_ids) / BATCH_SIZE / GRAD_ACCUM)
total_steps = steps_per_epoch * EPOCHS
warmup_steps = max(1, int(total_steps * 0.1))
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lambda s: min(1.0, s / warmup_steps) * max(0.0, (total_steps - s) / max(1, total_steps - warmup_steps))
)
scaler = torch.amp.GradScaler("cuda", enabled=True)

best_val_loss = float("inf")
best_state = None
epochs_no_improve = 0
history = []

print(f"Training: {EPOCHS} max epochs, lr={LR}, batch={BATCH_SIZE}x{GRAD_ACCUM} "
      f"(effective {BATCH_SIZE*GRAD_ACCUM}), KD temperature={KD_TEMPERATURE}, alpha={KD_ALPHA}")

for epoch in range(1, EPOCHS + 1):
    student.train()
    t0 = time.time()
    running_loss = running_ce = running_kl = 0.0
    n_batches = 0
    optimizer.zero_grad(set_to_none=True)

    for i, (input_ids, attention_mask, labels) in enumerate(make_batches(train_ids, train_labels, BATCH_SIZE, shuffle=True, seed=SEED + epoch)):
        input_ids, attention_mask, labels = input_ids.to(DEVICE), attention_mask.to(DEVICE), labels.to(DEVICE)

        with torch.no_grad():
            teacher_out = teacher(input_ids=input_ids, attention_mask=attention_mask)
            teacher_logits = teacher_out.logits

        with torch.amp.autocast("cuda", dtype=torch.float16):
            student_out = student(input_ids=input_ids, attention_mask=attention_mask)
            loss, ce_val, kl_val = distillation_loss(student_out.logits, teacher_logits, labels)

        scaler.scale(loss / GRAD_ACCUM).backward()
        running_loss += loss.item(); running_ce += ce_val.item(); running_kl += kl_val.item()
        n_batches += 1

        if (i + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

    avg_train_loss = running_loss / max(1, n_batches)
    avg_train_ce = running_ce / max(1, n_batches)
    avg_train_kl = running_kl / max(1, n_batches)

    student.eval()
    val_loss_sum = val_ce_sum = val_kl_sum = 0.0
    n_val = 0
    with torch.no_grad():
        for input_ids, attention_mask, labels in make_batches(val_ids, val_labels, BATCH_SIZE, shuffle=False):
            input_ids, attention_mask, labels = input_ids.to(DEVICE), attention_mask.to(DEVICE), labels.to(DEVICE)
            teacher_logits = teacher(input_ids=input_ids, attention_mask=attention_mask).logits
            with torch.amp.autocast("cuda", dtype=torch.float16):
                student_logits = student(input_ids=input_ids, attention_mask=attention_mask).logits
                v_loss, v_ce, v_kl = distillation_loss(student_logits, teacher_logits, labels)
            val_loss_sum += v_loss.item(); val_ce_sum += v_ce.item(); val_kl_sum += v_kl.item()
            n_val += 1
    avg_val_loss = val_loss_sum / max(1, n_val)
    avg_val_ce = val_ce_sum / max(1, n_val)
    avg_val_kl = val_kl_sum / max(1, n_val)

    history.append({"epoch": epoch, "train_loss": avg_train_loss, "train_ce": avg_train_ce, "train_kl": avg_train_kl,
                    "val_loss": avg_val_loss, "val_ce": avg_val_ce, "val_kl": avg_val_kl, "seconds": time.time() - t0})
    print(f"  epoch {epoch}/{EPOCHS}  train_loss={avg_train_loss:.4f} (ce={avg_train_ce:.4f} kl={avg_train_kl:.4f})  "
          f"val_loss={avg_val_loss:.4f} (ce={avg_val_ce:.4f} kl={avg_val_kl:.4f})  ({time.time()-t0:.0f}s)")

    if not math.isfinite(avg_val_loss):
        raise RuntimeError(f"validation loss is not finite at epoch {epoch}; training diverged")

    if avg_val_loss < best_val_loss - 1e-4:
        best_val_loss = avg_val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in student.state_dict().items()}
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= 3:
            print(f"  Early stopping at epoch {epoch} (no improvement for 3 epochs)")
            break

    gc.collect()
    torch.cuda.empty_cache()

student.load_state_dict(best_state)
student.save_pretrained(str(BEST_MODEL_DIR))
tokenizer.save_pretrained(str(BEST_MODEL_DIR))
(WORK / "train_history.json").write_text(json.dumps(history, indent=1))
print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}. Saved to {BEST_MODEL_DIR}")

del teacher
gc.collect()
torch.cuda.empty_cache()
print(f"Teacher released from memory. GPU free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

## Reload the trained student for evaluation

In [ ]:
# CELL 7 — RELOAD THE TRAINED STUDENT FOR EVALUATION (teacher already released from memory)
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(str(BEST_MODEL_DIR))
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"   # required for correct batched generation with a decoder-only model

model = AutoModelForCausalLM.from_pretrained(str(BEST_MODEL_DIR)).to(DEVICE)
model.eval()
print(f"Reloaded trained student from {BEST_MODEL_DIR} onto {DEVICE}")

## Evaluation kit (subprocess-isolated, hard timeouts, proven against reference solutions)

In [ ]:
# CELL 8 — EVALUATION KIT (subprocess-isolated, hard timeouts, proven against reference solutions)
PRELUDE = ("import resource\n"
           "try:\n    resource.setrlimit(resource.RLIMIT_AS, (3 << 30, 3 << 30))\nexcept Exception:\n    pass\n")

def run_python(files, entry, args=(), timeout=10):
    with tempfile.TemporaryDirectory() as d:
        for name, text in files.items():
            Path(d, name).write_text((PRELUDE + text) if name == entry else text, encoding="utf-8")
        proc = subprocess.Popen([sys.executable, entry, *map(str, args)], cwd=d, stdin=subprocess.DEVNULL,
                                stdout=subprocess.PIPE, stderr=subprocess.PIPE, encoding="utf-8",
                                errors="replace", start_new_session=True)
        try:
            out, err = proc.communicate(timeout=timeout)
            return proc.returncode, out, err
        except subprocess.TimeoutExpired:
            try: os.killpg(proc.pid, signal.SIGKILL)
            except Exception: pass
            try: proc.communicate(timeout=5)
            except Exception: pass
            return -9, "", "timeout"

HE_RUNNER = '''
import json, sys
spec = json.load(open("spec.json"))
src = open("prog.py", encoding="utf-8").read()
ns = {"__name__": "heval_module"}
try:
    exec(compile(src, "prog.py", "exec"), ns)
    ns["check"](ns[spec["entry"]])
except BaseException as e:
    sys.stderr.write(type(e).__name__ + ": " + str(e)[:300] + "\\n")
    sys.exit(1)
print("__PASS__")
'''

CE_RUNNER = '''
import io, json, sys, unittest
src = open("gen.py", encoding="utf-8").read() + "\\n\\n" + open("test.py", encoding="utf-8").read()
ns = {"__name__": "classeval_module"}
try:
    exec(compile(src, "prog.py", "exec"), ns)
except BaseException as e:
    print(json.dumps({"status": "exec_fail", "error": (type(e).__name__ + ": " + str(e))[:200]})); sys.exit(0)
classes = [v for v in ns.values() if isinstance(v, type) and issubclass(v, unittest.TestCase) and v is not unittest.TestCase]
total = failed = 0
for cls in classes:
    res = unittest.TextTestRunner(stream=io.StringIO(), verbosity=0).run(unittest.defaultTestLoader.loadTestsFromTestCase(cls))
    total += res.testsRun
    failed += len(res.failures) + len(res.errors)
print(json.dumps({"status": "ok", "tests": total, "failed": failed}))
'''

def top_level_imports(src):
    try:
        tree = ast.parse(src)
    except SyntaxError:
        return "\n".join(l for l in src.splitlines() if l.startswith(("import ", "from ")))
    keep = [n for n in tree.body if isinstance(n, (ast.Import, ast.ImportFrom))]
    return ast.unparse(ast.Module(body=keep, type_ignores=[])) if keep else ""

def humaneval_header(prompt, entry):
    try:
        tree = ast.parse(prompt)
        keep = [n for n in tree.body if isinstance(n, (ast.Import, ast.ImportFrom))
                or (isinstance(n, ast.FunctionDef) and n.name != entry)]
        return ast.unparse(ast.Module(body=keep, type_ignores=[])) if keep else ""
    except SyntaxError:
        return top_level_imports(prompt)

def run_humaneval(prompt, entry, test, gen_src, timeout=20):
    prog = humaneval_header(prompt, entry) + "\n\n" + gen_src + "\n\n" + test + "\n"
    rc, out, err = run_python({"runner.py": HE_RUNNER, "prog.py": prog, "spec.json": json.dumps({"entry": entry})},
                              "runner.py", timeout=timeout)
    return rc == 0 and "__PASS__" in out

def run_classeval(skeleton, test, gen_src, timeout=30):
    gen = top_level_imports(skeleton) + "\n\n" + gen_src
    rc, out, err = run_python({"runner.py": CE_RUNNER, "gen.py": gen, "test.py": test}, "runner.py", timeout=timeout)
    if rc == -9:
        return {"status": "timeout", "tests": 0, "failed": 0}
    try:
        return json.loads(out.strip().splitlines()[-1])
    except Exception:
        return {"status": "runner_error", "tests": 0, "failed": 0}

def is_valid_python(code_str):
    try:
        tree = ast.parse(code_str)
    except Exception:
        return False
    try:
        compile(tree, "<e>", "exec")
        return True
    except Exception:
        return False

def defines(src, name, kind):
    try:
        return any(isinstance(n, kind) and n.name == name for n in ast.walk(ast.parse(src)))
    except Exception:
        return False

def canonical_humaneval_src(row):
    tree = ast.parse(row["prompt"] + row["canonical_solution"])
    node = next(n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name == row["entry_point"])
    return ast.unparse(node)

def load_humaneval_rows(source):
    if source == "hf":
        from datasets import load_dataset
        return [dict(r) for r in load_dataset("evalplus/humanevalplus", split="test")]
    import gzip, urllib.request
    raw = urllib.request.urlopen("https://raw.githubusercontent.com/openai/human-eval/master/data/HumanEval.jsonl.gz", timeout=60).read()
    return [json.loads(l) for l in gzip.decompress(raw).decode("utf-8").splitlines() if l.strip()]

def load_classeval_rows(source):
    if source == "hf":
        from datasets import load_dataset
        return [dict(r) for r in load_dataset("FudanSELab/ClassEval", split="test")]
    import urllib.request
    return json.loads(urllib.request.urlopen("https://raw.githubusercontent.com/FudanSELab/ClassEval/master/data/ClassEval_data.json", timeout=60).read().decode("utf-8"))

def select_humaneval(n, seed):
    best = None
    for source in ("hf", "github"):
        try:
            rows = load_humaneval_rows(source)
        except Exception as e:
            print(f"  HumanEval from {source}: unavailable ({type(e).__name__}: {str(e)[:80]})")
            continue
        idx = list(range(len(rows)))
        random.Random(seed).shuffle(idx)
        step = len(idx) / n
        chosen = [rows[i] for i in sorted(idx[int(k * step)] for k in range(n))]
        good = []
        for r in chosen:
            try:
                ok = run_humaneval(r["prompt"], r["entry_point"], r["test"], canonical_humaneval_src(r), timeout=20)
            except Exception as e:
                print(f"    reference-check error on {r.get('task_id')}: {type(e).__name__}: {str(e)[:100]}")
                ok = False
            if ok:
                good.append(r)
        print(f"  HumanEval from {source}: {len(good)}/{len(chosen)} reference solutions pass the harness")
        if best is None or len(good) > len(best[1]):
            best = (source, good, len(chosen))
        if len(good) >= 0.9 * len(chosen):
            break
    return best

def select_classeval(n, seed, tok):
    for source in ("hf", "github"):
        try:
            rows = load_classeval_rows(source)
            break
        except Exception as e:
            print(f"  ClassEval from {source}: unavailable ({type(e).__name__}: {str(e)[:80]})")
    else:
        return None
    fits = [r for r in rows if len(tok(f"<GENERATE> Complete this Python class:\n{r['skeleton']}")["input_ids"]) <= CLASSEVAL_MAX_PROMPT_TOKENS]
    random.Random(seed).shuffle(fits)
    chosen, tried = [], 0
    for r in fits:
        if len(chosen) >= n or tried >= 60:
            break
        tried += 1
        try:
            res = run_classeval(r["skeleton"], r["test"], r["solution_code"], timeout=30)
        except Exception as e:
            print(f"    reference-check error on {r.get('task_id')}: {type(e).__name__}: {str(e)[:100]}")
            res = {"status": "error"}
        if res.get("status") == "ok" and res.get("tests", 0) > 0 and res.get("failed", 1) == 0:
            chosen.append(r)
    print(f"  ClassEval from {source}: {len(rows)} problems, {len(fits)} fit the encoder window, "
          f"tried {tried}, kept {len(chosen)} whose reference solution passes the harness")
    return source, chosen

print("Evaluation kit ready (isolated processes, hard time limits, tests must actually run).")

## Generation: greedy pass@1 (primary) + compile-validated (separate, not pass@1)

In [ ]:
# CELL 9 — GENERATION (decoder-only: slice off the echoed prompt). Same two-mechanism split as
# every other notebook in this project: greedy is the primary, honest pass@1; compile-validated is
# separate and explicitly NOT a correctness metric.
@torch.no_grad()
def generate_texts_greedy(mdl, prompts, max_new_tokens=400, batch_size=GEN_BATCH):
    mdl.eval()
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
    outs = [None] * len(prompts)
    for s in range(0, len(order), batch_size):
        idx = order[s:s + batch_size]
        enc = tokenizer([prompts[i] for i in idx], return_tensors="pt", padding=True,
                        truncation=True, max_length=MAX_SEQ_LEN).to(DEVICE)
        input_len = enc["input_ids"].shape[1]
        with torch.amp.autocast("cuda", enabled=True):
            seq = mdl.generate(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"],
                               max_new_tokens=max_new_tokens, num_beams=1, do_sample=False,
                               pad_token_id=tokenizer.pad_token_id)
        for j, i in enumerate(idx):
            outs[i] = tokenizer.decode(seq[j][input_len:], skip_special_tokens=True)
    return outs

@torch.no_grad()
def generate_compile_validated(mdl, prompt, max_new_tokens=400, num_beams=4):
    enc = tokenizer([prompt], return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN).to(DEVICE)
    input_len = enc["input_ids"].shape[1]
    with torch.amp.autocast("cuda", enabled=True):
        seq = mdl.generate(**enc, max_new_tokens=max_new_tokens, num_beams=num_beams, early_stopping=True,
                           pad_token_id=tokenizer.pad_token_id)
    out = tokenizer.decode(seq[0][input_len:], skip_special_tokens=True)
    if is_valid_python(out):
        return out, "beam"
    for temp in RETRY_TEMPERATURES:
        with torch.amp.autocast("cuda", enabled=True):
            seq2 = mdl.generate(**enc, max_new_tokens=max_new_tokens, do_sample=True, temperature=temp,
                                top_p=0.95, pad_token_id=tokenizer.pad_token_id)
        out2 = tokenizer.decode(seq2[0][input_len:], skip_special_tokens=True)
        if is_valid_python(out2):
            return out2, f"sample_t{temp}"
    return out, "beam_invalid"

print("Generation functions ready: generate_texts_greedy (pass@1) and generate_compile_validated (NOT pass@1).")

## Run the full evaluation

In [ ]:
# CELL 10 — RUN THE FULL EVALUATION
summary = {}

print("Selecting benchmark problems (reference solutions run through the harness first)...")
he_sel = None
try:
    he_sel = select_humaneval(HE_N, SEED)
except Exception as e:
    print(f"  HumanEval selection failed ({type(e).__name__}: {str(e)[:150]})")
he_problems = he_sel[1] if he_sel else []

ce_sel = None
try:
    ce_sel = select_classeval(CLASSEVAL_N, SEED, tokenizer)
except Exception as e:
    print(f"  ClassEval selection failed ({type(e).__name__}: {str(e)[:150]})")
ce_problems = ce_sel[1] if ce_sel else []
print(f"\nUsable: HumanEval {len(he_problems)}/{HE_N}  ClassEval {len(ce_problems)}/{CLASSEVAL_N}")

random.seed(SEED)
test_df = df[df.split == "test"].reset_index(drop=True)
eval_rows = test_df.sample(min(EVAL_OWN_N, len(test_df)), random_state=SEED).reset_index(drop=True)
print(f"\nGenerating {len(eval_rows)} own-dataset outputs (greedy, pass@1)...")
t0 = time.time()
gens = generate_texts_greedy(model, eval_rows["prompt_text"].tolist())
gen_sec = time.time() - t0
print(f"  done in {gen_sec:.0f}s")

def norm_dump(code_str):
    tree = ast.parse(code_str)
    for n in ast.walk(tree):
        if isinstance(n, ast.Constant):
            n.kind = None
        if hasattr(n, "type_comment"):
            n.type_comment = None
    return ast.dump(tree)

def exact_match(pred_src, ref_src):
    try:
        return norm_dump(pred_src) == norm_dump(ref_src)
    except Exception:
        return False

own_records = []
for (_, row), text in zip(eval_rows.iterrows(), gens):
    valid = is_valid_python(text)
    exact = exact_match(text, row[TARGET_FORMAT_COL]) if valid else False
    own_records.append({"id": row["id"], "task": row["task"], "valid": valid, "exact": exact})
own_df = pd.DataFrame(own_records)

summary["own_greedy"] = {
    "n": len(own_df), "valid": int(own_df.valid.sum()), "exact": int(own_df.exact.sum()), "gen_seconds": gen_sec,
    "by_task": {t: {"n": len(g), "valid": int(g.valid.sum()), "exact": int(g.exact.sum())}
                for t, g in own_df.groupby("task")},
}
print(f"Own-dataset greedy: valid {summary['own_greedy']['valid']}/{summary['own_greedy']['n']}  "
      f"exact {summary['own_greedy']['exact']}/{summary['own_greedy']['n']}")
for t, s in summary["own_greedy"]["by_task"].items():
    print(f"  {t:9s} valid {s['valid']}/{s['n']}  exact {s['exact']}/{s['n']}")

def he_prompt(p):
    return f"{SYSTEM_PROMPT}\n\n<GENERATE> Complete this Python function:\n{p}\n<OUTPUT>\n"
def ce_prompt(skel):
    return f"{SYSTEM_PROMPT}\n\n<GENERATE> Complete this Python class:\n{skel}\n<OUTPUT>\n"

he_passed = 0
if he_problems:
    gens_he = generate_texts_greedy(model, [he_prompt(r["prompt"]) for r in he_problems])
    for r, text in zip(he_problems, gens_he):
        code = text if (is_valid_python(text) and defines(text, r["entry_point"], ast.FunctionDef)) else r["prompt"] + text
        has_def = is_valid_python(code) and defines(code, r["entry_point"], ast.FunctionDef)
        passed = run_humaneval(r["prompt"], r["entry_point"], r["test"], code) if has_def else False
        he_passed += bool(passed)
    print(f"\nHumanEval pass@1 (greedy): {he_passed}/{len(he_problems)} ({he_passed/len(he_problems)*100:.1f}%)")
else:
    print("\nHumanEval: skipped (no usable problems)")
summary["humaneval"] = {"n": len(he_problems), "passed": he_passed}

ce_passed = 0
if ce_problems:
    gens_ce = generate_texts_greedy(model, [ce_prompt(r["skeleton"]) for r in ce_problems])
    for r, text in zip(ce_problems, gens_ce):
        has_cls = is_valid_python(text) and defines(text, r["class_name"], ast.ClassDef)
        res = run_classeval(r["skeleton"], r["test"], text) if has_cls else {"status": "no_class", "tests": 0, "failed": 0}
        passed = res.get("status") == "ok" and res.get("tests", 0) > 0 and res.get("failed", 1) == 0
        ce_passed += bool(passed)
    print(f"ClassEval class-level pass@1 (greedy): {ce_passed}/{len(ce_problems)} ({ce_passed/len(ce_problems)*100:.1f}%)")
else:
    print("ClassEval: skipped (no usable problems)")
summary["classeval"] = {"n": len(ce_problems), "passed": ce_passed}

print("\n" + "=" * 70)
print("COMPILE-VALIDATED GENERATION (beam + retry-until-valid -- NOT a pass@1/correctness metric)")
print("=" * 70)
he_cv_valid = 0
if he_problems:
    for r in he_problems:
        text, method = generate_compile_validated(model, he_prompt(r["prompt"]))
        he_cv_valid += is_valid_python(text)
    print(f"HumanEval prompts: valid Python under compile-validated generation: {he_cv_valid}/{len(he_problems)} "
          f"({he_cv_valid/len(he_problems)*100:.1f}%)")
summary["humaneval_compile_validated_valid_pct"] = (he_cv_valid / len(he_problems) * 100) if he_problems else None

RESULTS_FILE.write_text(json.dumps(summary, indent=2, default=str))
print(f"\nSaved: {RESULTS_FILE}")

## Report — compare against the plain-SFT Qwen 0.5B run

In [ ]:
# CELL 11 — REPORT: DID DISTILLATION HELP BEYOND PLAIN FINE-TUNING?
history = json.loads((WORK / "train_history.json").read_text())
summary = json.loads(RESULTS_FILE.read_text())

print("=" * 90)
print(f"Qwen2.5-Coder-0.5B DISTILLED from Qwen2.5-Coder-7B-Instruct (single GPU)")
print("=" * 90)

print(f"\nLoss by epoch (train_loss -> val_loss, each split into ce/kl components):")
for h in history:
    print(f"  epoch {h['epoch']}: train={h['train_loss']:.4f} (ce={h['train_ce']:.4f} kl={h['train_kl']:.4f})  "
          f"val={h['val_loss']:.4f} (ce={h['val_ce']:.4f} kl={h['val_kl']:.4f})")

og = summary["own_greedy"]
print(f"\nOwn-dataset, GREEDY pass@1 (n={og['n']}):")
print(f"  valid Python: {og['valid']}/{og['n']} ({og['valid']/og['n']*100:.1f}%)")
print(f"  exact match:  {og['exact']}/{og['n']} ({og['exact']/og['n']*100:.1f}%)")
for t, s in og["by_task"].items():
    print(f"    {t:9s} valid {s['valid']}/{s['n']} ({s['valid']/s['n']*100:.1f}%)   "
          f"exact {s['exact']}/{s['n']} ({s['exact']/s['n']*100:.1f}%)")

he, ce = summary["humaneval"], summary["classeval"]
print(f"\nHumanEval pass@1 (greedy): {he['passed']}/{he['n']}" + (f" ({he['passed']/he['n']*100:.1f}%)" if he["n"] else " (skipped)"))
print(f"ClassEval pass@1 (greedy): {ce['passed']}/{ce['n']}" + (f" ({ce['passed']/ce['n']*100:.1f}%)" if ce["n"] else " (skipped)"))

print("\n" + "=" * 90)
print("COMPARISON: PLAIN FINE-TUNING vs. DISTILLATION (same base model, same dataset)")
print("=" * 90)
gen = og["by_task"].get("generate", {"n": 1, "valid": 0, "exact": 0})
rep = og["by_task"].get("repair", {"n": 1, "valid": 0, "exact": 0})
print(f"{'metric':<22} {'Qwen0.5B plain SFT':>20} {'Qwen0.5B DISTILLED':>20}")
print(f"{'generate valid %':<22} {'95.0':>20} {gen['valid']/gen['n']*100:>20.1f}")
print(f"{'generate exact %':<22} {'0.8':>20} {gen['exact']/gen['n']*100:>20.1f}")
print(f"{'repair exact %':<22} {'59.7':>20} {rep['exact']/rep['n']*100:>20.1f}")
he_pct = (he["passed"] / he["n"] * 100) if he["n"] else 0
print(f"{'HumanEval %':<22} {'17.2':>20} {he_pct:>20.1f}")
ce_pct_val = (ce["passed"] / ce["n"] * 100) if ce["n"] else 0
print(f"{'ClassEval %':<22} {'10.0':>20} {ce_pct_val:>20.1f}")

print("\n" + "=" * 90)
print("VERDICT")
print("=" * 90)
if he_pct > 17.2 + 3 or (gen["exact"] / gen["n"] * 100) > 0.8 + 3:
    print("Distillation meaningfully improved on plain fine-tuning -- the teacher's soft-label")
    print("guidance transferred real additional capability to the student. This recipe is worth")
    print("scaling: more epochs, a larger/more diverse distillation dataset, and/or trying on-policy")
    print("distillation (student generates, teacher scores) as a documented further improvement.")
elif he_pct < 17.2 - 3:
    print("Distillation UNDERPERFORMED plain fine-tuning on HumanEval. Possible causes to check:")
    print("  - alpha/temperature may need tuning (KD_ALPHA and KD_TEMPERATURE in Cell 1)")
    print("  - the teacher's own quality on these specific tasks may be weaker than expected")
    print("  - batch=1 training (forced by memory) may be noisier than the plain SFT run's batch=2")
else:
    print("Distillation performed about the same as plain fine-tuning -- the teacher's soft labels")
    print("did not add a clear, measurable advantage on this dataset at these hyperparameters.")
    print("Worth trying: a different alpha/temperature, a stronger or more specialized teacher, or")
    print("on-policy distillation instead of the off-policy (teacher-forced) version used here.")

## Finish (download from Kaggle's Output tab)

In [ ]:
# CELL 12 — FINISH (Kaggle: download from the Output tab)
for f in (RESULTS_FILE, WORK / "train_history.json"):
    print(f"{f}  ({f.stat().st_size/1e3:.0f} KB)")
print(f"Model checkpoint: {BEST_MODEL_DIR}  (saved, large -- see note below)")
print("\nTo get these files onto your computer:")
print("  1. Click 'Save Version' (top right) -> 'Save & Run All' -> Save.")
print("  2. Once it finishes, open this notebook's 'Output' tab.")
print("  3. results.json and train_history.json will be listed there with a Download button.")
print("  Note: the saved model checkpoint directory is large; download only results.json and")
print("  train_history.json unless you specifically need the model weights.")